# Test — inspeccionar un .tif (imagen 4-band + máscara)

Verifica que la imagen Sentinel tiene **4 bandas (R, G, B, NIR)** y que la máscara es binaria (0/1). Lee de Drive.

In [ ]:
!pip install -q rasterio

from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/deforestacion-peru")
except ImportError:
    BASE_DIR = Path(".")

DATA_DIR = BASE_DIR / "data"
IMG_DIR  = DATA_DIR / "sentinel_rgbn"
MASK_DIR = DATA_DIR / "masks"

ROW_ID = 9908   # cambia este id para inspeccionar otro par
img_path  = IMG_DIR  / f"s2_rgbn_{ROW_ID}.tif"
mask_path = MASK_DIR / f"mask_{ROW_ID}.tif"
print("imagen :", img_path, "| existe:", img_path.exists())
print("máscara:", mask_path, "| existe:", mask_path.exists())

## 1. Imagen — cuántas bandas y rango de la NIR

In [ ]:
import rasterio

with rasterio.open(img_path) as s:
    print("bandas:", s.count, "| tamaño:", s.width, "x", s.height, "| dtype:", s.dtypes[0])
    for b in range(1, s.count + 1):
        arr = s.read(b)
        print(f"  banda {b}: min={arr.min()} max={arr.max()} media={arr.mean():.0f}")
    if s.count >= 4:
        nir = s.read(4)
        print("\nvalores banda 4 (NIR):", nir.min(), nir.max())
    else:
        print("\n⚠️ este tif NO tiene 4 bandas — falta la NIR")

## 2. Ejemplo de un píxel — los 4 valores crudos y normalizados

Reflectancia guardada como entero ×10000. Dividiendo por 10000 vuelve a [0,1].

In [ ]:
import numpy as np

with rasterio.open(img_path) as s:
    img = s.read().astype("float32")   # (bandas, H, W)

r, c = img.shape[1] // 2, img.shape[2] // 2   # píxel del centro
nombres = ["R", "G", "B", "NIR"]
print(f"píxel ({r},{c}):")
for i in range(img.shape[0]):
    crudo = img[i, r, c]
    nombre = nombres[i] if i < len(nombres) else f"banda{i+1}"
    print(f"  {nombre:4s}: crudo={crudo:7.0f}  ->  norm={crudo/10000:.3f}")

## 3. Máscara — confirmar que es binaria (0/1)

In [ ]:
with rasterio.open(mask_path) as s:
    m = s.read(1)
    print("bandas:", s.count, "| tamaño:", s.width, "x", s.height, "| dtype:", s.dtypes[0])
    vals, counts = np.unique(m, return_counts=True)
    print("valores únicos:", vals.tolist())
    total = m.size
    for v, n in zip(vals, counts):
        print(f"  valor {v}: {n} píxeles ({100*n/total:.1f}%)")

## 4. Ver imagen RGB + NIR + máscara

In [ ]:
import matplotlib.pyplot as plt

def stretch(a, lo=2, hi=98):
    a = a.astype("float32"); pl, ph = np.percentile(a, [lo, hi])
    return np.clip((a - pl) / (ph - pl + 1e-6), 0, 1)

with rasterio.open(img_path) as s:
    img = s.read().astype("float32")
with rasterio.open(mask_path) as s:
    mask = s.read(1)

rgb = np.dstack([stretch(img[c]) for c in range(3)])
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
ax[0].imshow(rgb); ax[0].set_title("RGB (bandas 1-3)")
if img.shape[0] >= 4:
    ax[1].imshow(stretch(img[3]), cmap="gray"); ax[1].set_title("NIR (banda 4)")
else:
    ax[1].set_title("sin banda NIR")
ax[2].imshow(rgb); ax[2].imshow(mask, cmap="Reds", alpha=0.45, vmin=0, vmax=1)
ax[2].set_title("máscara (rojo = deforestado)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()